# Dataset v6.1 — COVID-19 UCI: Mortalidad a 30 días

## Descripción

**v6.1** combina los criterios de inclusión amplios de v6 con las features enriquecidas de v7.

### Criterio de inclusión (v6)
- Todos los pacientes UCI con diagnóstico COVID-19 (ICD-10: `U071`) en **cualquier posición** `seq_num`.
- Sin restricción de diagnóstico principal.
- Equivale al criterio original de v6 (~3 908 estancias).

### Features (v7)
- Ventana de extracción de **72 horas** desde el ingreso UCI.
- **5 agregaciones** por variable: `val_first`, `val_last`, `val_min`, `val_max`, `val_delta`.
- **22 variables PMHX** (comorbilidades binarias).
- **24 variables binarias clínicas** (`bin_*`).
- SOFA parcial, APACHE II parcial, severity score.

### Objetivo
Comparar el impacto de los criterios de inclusión (v6 vs v7) manteniendo la arquitectura de features constante (v7).


## 1. Setup

In [1]:
import os
import sys
os.environ['_JAVA_OPTIONS'] = '-Djava.security.manager=allow -Duser.name=julianromero'
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from pyspark.sql import SparkSession
import pyspark.pandas as ps

spark = SparkSession.builder.appName('SparkSession').getOrCreate()

from utils.sql_manager import SQLManager
sql_manager = SQLManager(queries_dir='../queries')

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pyspark/pandas/__init__.py:43: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(
Picked up _JAVA_OPTIONS: -Djava.security.manager=allow -Duser.name=julianromero
Picked up _JAVA_OPTIONS: -Djava.security.manager=allow -Duser.name=julianromero
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/18 12:44:49 WARN Utils: Your hostname, MacBook-Air-de-Julian.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.80 instead (on interface en0)
26/04/18 12:44:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting

## 2. Carga de archivos raw

In [2]:
hosp_adm         = spark.read.csv('../data/nw_hosp/admissions.csv',     header=True, inferSchema=True)
icu_stays        = spark.read.csv('../data/nw_icu/icustays.csv',        header=True, inferSchema=True)
patients         = spark.read.csv('../data/nw_hosp/patients.csv',       header=True, inferSchema=True)
chart_events     = spark.read.csv('../data/nw_icu/chartevents.csv',     header=True, inferSchema=True)
lab_events       = spark.read.csv('../data/nw_hosp/labevents.csv',      header=True, inferSchema=True)
procedure_events = spark.read.csv('../data/nw_icu/procedureevents.csv', header=True, inferSchema=True)
diagnoses_icd    = spark.read.csv('../data/nw_hosp/diagnoses_icd.csv',  header=True, inferSchema=True)

hosp_adm.createOrReplaceTempView('admissions')
icu_stays.createOrReplaceTempView('icu_stays')
patients.createOrReplaceTempView('patients')
chart_events.createOrReplaceTempView('chart_events')
lab_events.createOrReplaceTempView('lab_events')
procedure_events.createOrReplaceTempView('procedure_events')
diagnoses_icd.createOrReplaceTempView('diagnoses_icd')

## 3. Queries SQL iniciales

In [3]:
hosp_adm_data = sql_manager.execute(spark, 'initial_data/admissions.sql')
hosp_adm_data.createOrReplaceTempView('hosp_adm_data')

icu_stays_data = sql_manager.execute(spark, 'initial_data/icu_stays.sql')
icu_stays_data.createOrReplaceTempView('icu_stays_data')

# chart_events_data y lab_events_data: vistas con limpieza mínima usadas por los joins
chart_events_data = sql_manager.execute(spark, 'initial_data/chart_events.sql')
chart_events_data.createOrReplaceTempView('chart_events_data')

patients_data = sql_manager.execute(spark, 'initial_data/patients_filtered.sql')
patients_data.createOrReplaceTempView('patients_data')

lab_events_data = sql_manager.execute(spark, 'initial_data/lab_events.sql')
lab_events_data.createOrReplaceTempView('lab_events_data')

procedure_events_data = sql_manager.execute(spark, 'initial_data/procedure_events.sql')
procedure_events_data.createOrReplaceTempView('procedure_events_data')

## 4. [NUEVO v7] Joins con ventana 72h

Se sustituyen los joins de 24h (`date_add(..., 1)`) por ventana de **72h** (`date_add(..., 3)`).

Se incluye `charttime` en los outputs para poder calcular primer/último valor.

> Para `procedure_events_icu` se crea la vista con el **mismo nombre** que espera `procedure_flags.sql`, así puede reutilizarse sin modificar el archivo SQL.

In [4]:
# ── Chart events: ventana 72h, incluye charttime ─────────────────────────────
chart_events_icu_72h = spark.sql("""
    SELECT
        a.subject_id, a.admission_id, a.stay_id,
        c.itemid, c.value, c.charttime
    FROM icu_stays_data a
    INNER JOIN chart_events_data c
        ON a.stay_id = c.stay_id
        AND c.charttime >= a.admittime
        AND c.charttime <= date_add(a.admittime, 3)
""")
chart_events_icu_72h.createOrReplaceTempView('chart_events_icu_72h')

# ── Lab events: ventana 72h, incluye charttime ───────────────────────────────
lab_events_icu_72h = spark.sql("""
    SELECT
        a.subject_id, a.admission_id, a.stay_id,
        l.itemid, l.value_num, l.charttime
    FROM icu_stays_data a
    INNER JOIN lab_events_data l
        ON a.admission_id = l.admission_id
        AND l.charttime >= a.admittime
        AND l.charttime <= date_add(a.admittime, 3)
""")
lab_events_icu_72h.createOrReplaceTempView('lab_events_icu_72h')

# ── Procedure events: ventana 72h — mismo nombre de vista que procedure_flags.sql ─
procedure_events_icu_72h = spark.sql("""
    SELECT
        a.subject_id, a.admission_id, a.stay_id,
        c.itemid, c.value, c.starttime
    FROM icu_stays_data a
    INNER JOIN procedure_events_data c
        ON a.stay_id = c.stay_id
        AND c.starttime >= a.admittime
        AND c.starttime <= date_add(a.admittime, 3)
""")
# Crear con el nombre que espera procedure_flags.sql
procedure_events_icu_72h.createOrReplaceTempView('procedure_events_icu')

print(f'chart_events_icu_72h:     {chart_events_icu_72h.count()} registros')
print(f'lab_events_icu_72h:       {lab_events_icu_72h.count()} registros')
print(f'procedure_events_icu_72h: {procedure_events_icu_72h.count()} registros')

chart_events_icu_72h:     5352930 registros


lab_events_icu_72h:       2583815 registros


procedure_events_icu_72h: 601947 registros


## 5. Definición de variables y helpers SQL

Se definen los itemids y nombres cortos para labs y vitales, y se generan las queries SQL dinámicamente.

**Convención de nombres:** `{lab|vital}_{nombre}_val_{first|last|min|max}` en Spark; `_val_delta` se añade en pandas.

In [5]:
# ── Variables de laboratorio: (itemid, nombre_corto) ─────────────────────────
LAB_ITEMS = [
    (100001, 'glucose'),
    (100002, 'creatinine'),
    (100004, 'bun'),
    (100006, 'hematocrit'),
    (100010, 'sodium'),
    (100011, 'potassium'),
    (100014, 'platelets'),
    (100016, 'wbc'),
    (100020, 'bilirubin'),
    (100022, 'neutrophils'),
    (100029, 'pao2'),
    (100031, 'lactate'),
    (100034, 'pt'),
    (100037, 'lymphocytes'),
    (100052, 'ferritin'),
    (100053, 'crp'),
    (100059, 'troponin'),
    (100060, 'peep'),
    (100075, 'dimer'),
    (100077, 'tidal_volume'),
    (100339, 'ph'),
]

# ── Signos vitales: (itemid, nombre_corto, rango_min, rango_max) ─────────────
# Los rangos se aplican antes del ranking para filtrar lecturas fisiológicamente inválidas
# sbp_line / dbp_line: solo para cálculo de PAM, se eliminan al final
CHART_ITEMS = [
    (323761, 'temp',     86,    113),   # Temperatura °F (30–45 °C)
    (320179, 'sbp',       0,   None),   # PA sistólica no invasiva
    (320180, 'dbp',       0,   None),   # PA diastólica no invasiva
    (320050, 'sbp_line',  0,   None),   # PA sistólica línea arterial (solo MAP)
    (320051, 'dbp_line',  0,   None),   # PA diastólica línea arterial (solo MAP)
    (320277, 'spo2',     50,    100),   # SpO₂ %
    (320045, 'hr',       20,    300),   # Frecuencia cardíaca lpm
    (320210, 'rr',        4,     70),   # Frecuencia respiratoria rpm
    (300001, 'bmi',      10,     80),   # BMI kg/m²
]

print(f'Variables de laboratorio: {len(LAB_ITEMS)}')
print(f'Signos vitales:           {len(CHART_ITEMS)}')
print(f'Total columnas base (×4 agg): {(len(LAB_ITEMS) + len(CHART_ITEMS)) * 4}')

Variables de laboratorio: 21
Signos vitales:           9
Total columnas base (×4 agg): 120


In [6]:
# ── Helpers para generar SQL de agregación ───────────────────────────────────

def gen_lab_agg_cases(items):
    """4 CASE WHEN bloques por lab: first, last, min, max."""
    lines = []
    for itemid, name in items:
        cond = f'itemid = {itemid}'
        lines += [
            f'    MAX(CASE WHEN {cond} AND rn_first = 1 THEN value_num END) AS lab_{name}_val_first',
            f'    MAX(CASE WHEN {cond} AND rn_last  = 1 THEN value_num END) AS lab_{name}_val_last',
            f'    MIN(CASE WHEN {cond} THEN value_num END)                  AS lab_{name}_val_min',
            f'    MAX(CASE WHEN {cond} THEN value_num END)                  AS lab_{name}_val_max',
        ]
    return ',\n'.join(lines)


def gen_chart_filter_condition(items):
    """Condición WHERE para filtrar vitales por rango clínico."""
    parts = []
    for itemid, name, lo, hi in items:
        c = f'itemid = {itemid}'
        if lo is not None: c += f' AND value_d >= {lo}'
        if hi is not None: c += f' AND value_d <= {hi}'
        parts.append(f'({c})')
    return '\n           OR '.join(parts)


def gen_chart_agg_cases(items):
    """4 CASE WHEN bloques por vital: first, last, min, max."""
    lines = []
    for itemid, name, lo, hi in items:
        cond = f'itemid = {itemid}'
        lines += [
            f'    MAX(CASE WHEN {cond} AND rn_first = 1 THEN value_d END) AS vital_{name}_val_first',
            f'    MAX(CASE WHEN {cond} AND rn_last  = 1 THEN value_d END) AS vital_{name}_val_last',
            f'    MIN(CASE WHEN {cond} THEN value_d END)                  AS vital_{name}_val_min',
            f'    MAX(CASE WHEN {cond} THEN value_d END)                  AS vital_{name}_val_max',
        ]
    return ',\n'.join(lines)


print('Helpers definidos: gen_lab_agg_cases, gen_chart_filter_condition, gen_chart_agg_cases')

Helpers definidos: gen_lab_agg_cases, gen_chart_filter_condition, gen_chart_agg_cases


## 6. Extracción de features de laboratorio (ventana 72h)

Una sola query con CTE de ranking calcula las 4 agregaciones para todos los labs en paralelo.

El **delta** (`last − first`) se calcula más adelante en pandas, tras cargar el parquet.

In [7]:
lab_cases_sql = gen_lab_agg_cases(LAB_ITEMS)

lab_features_v61 = spark.sql(f"""
    WITH lab_ranked AS (
        SELECT
            subject_id, admission_id, stay_id, itemid, value_num,
            ROW_NUMBER() OVER (
                PARTITION BY subject_id, admission_id, stay_id, itemid
                ORDER BY charttime ASC
            ) AS rn_first,
            ROW_NUMBER() OVER (
                PARTITION BY subject_id, admission_id, stay_id, itemid
                ORDER BY charttime DESC
            ) AS rn_last
        FROM lab_events_icu_72h
        WHERE value_num IS NOT NULL
    )
    SELECT
        subject_id, admission_id, stay_id,
{lab_cases_sql}
    FROM lab_ranked
    GROUP BY subject_id, admission_id, stay_id
""")

lab_features_v61.createOrReplaceTempView('lab_features_v61')
print(f'lab_features_v61: {lab_features_v61.count()} stays | {len(lab_features_v61.columns)} columnas')
print(f'  ({len(LAB_ITEMS)} labs × 4 agg = {len(LAB_ITEMS)*4} + 3 IDs = {len(LAB_ITEMS)*4+3} esperadas)')

26/04/18 12:45:20 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


lab_features_v7: 28020 stays | 87 columnas
  (21 labs × 4 agg = 84 + 3 IDs = 87 esperadas)


## 7. Extracción de signos vitales (ventana 72h)

Misma arquitectura de CTE con ranking. Los rangos clínicos se aplican **antes** del ranking
para garantizar que first/last correspondan a lecturas fisiológicamente válidas.

In [8]:
chart_filter_sql  = gen_chart_filter_condition(CHART_ITEMS)
chart_cases_sql   = gen_chart_agg_cases(CHART_ITEMS)

vital_features_v61 = spark.sql(f"""
    WITH chart_cast AS (
        -- Cast a double y filtro de nulos antes del ranking
        SELECT
            subject_id, admission_id, stay_id, itemid,
            try_cast(value AS DOUBLE) AS value_d,
            charttime
        FROM chart_events_icu_72h
        WHERE value IS NOT NULL
          AND try_cast(value AS DOUBLE) IS NOT NULL
    ),
    chart_filtered AS (
        -- Aplicar rangos clínicos por itemid
        SELECT * FROM chart_cast
        WHERE {chart_filter_sql}
    ),
    chart_ranked AS (
        SELECT
            subject_id, admission_id, stay_id, itemid, value_d,
            ROW_NUMBER() OVER (
                PARTITION BY subject_id, admission_id, stay_id, itemid
                ORDER BY charttime ASC
            ) AS rn_first,
            ROW_NUMBER() OVER (
                PARTITION BY subject_id, admission_id, stay_id, itemid
                ORDER BY charttime DESC
            ) AS rn_last
        FROM chart_filtered
    )
    SELECT
        subject_id, admission_id, stay_id,
{chart_cases_sql}
    FROM chart_ranked
    GROUP BY subject_id, admission_id, stay_id
""")

vital_features_v61.createOrReplaceTempView('vital_features_v61')
print(f'vital_features_v61: {vital_features_v61.count()} stays | {len(vital_features_v61.columns)} columnas')
print(f'  ({len(CHART_ITEMS)} vitales × 4 agg = {len(CHART_ITEMS)*4} + 3 IDs = {len(CHART_ITEMS)*4+3} esperadas)')

vital_features_v7: 20253 stays | 39 columnas
  (9 vitales × 4 agg = 36 + 3 IDs = 39 esperadas)


## 8. Flags de procedimientos (ventana 72h)

In [9]:
# procedure_flags.sql lee de 'procedure_events_icu', que ya apunta a los 72h (ver Sección 4)
procedure_flags = sql_manager.execute(spark, 'joins/procedure_flags.sql')
procedure_flags.createOrReplaceTempView('procedure_flags')
print(f'procedure_flags: {procedure_flags.count()} stays')

PROCEDURE_COLS = [
    'is_mechanical_ventilation', 'is_crrt', 'has_arterial_line',
    'is_intubated', 'is_prone_position', 'has_chest_tube',
    'is_dnr', 'has_central_line', 'has_hemodialysis',
    'has_niv', 'is_ecmo'
]

procedure_flags: 22809 stays


## 9. [NUEVO v7] Comorbilidades (PMHX) + criterios de inclusión

### Criterio de inclusión:
```
INCLUIR si:
  (A) icd_code = 'U071' AND seq_num = 1  ← COVID diagnóstico PRINCIPAL
  OR
  (B) icd_code = 'U071' AND seq_num = 2  ← COVID diagnóstico SECUNDARIO
      AND el dx principal (seq_num = 1) es una comorbilidad COVID-relacionada
```

> `seq_num` en MIMIC-IV es base 1: `seq_num = 1` = diagnóstico principal.

In [10]:
COMORBIDITY_DICT = {
    'pmhx_diabetes':         ['E09', 'E10', 'E11', 'E13'],
    'pmhx_hld':              ['E78'],
    'pmhx_htn':              ['I10', 'I11', 'I12', 'I13'],
    'pmhx_ihd':              ['I20', 'I21', 'I22', 'I25', 'Z95.1'],
    'pmhx_ckd':              ['N18', 'Z99.2', 'D63.1'],
    'pmhx_copd':             ['J44', 'J43', 'J41', 'J42', 'J47', 'J84', 'I27.0', 'I27.2', 'I26'],
    'pmhx_asthma':           ['J45'],
    'pmhx_activecancer':     ['C'],
    'pmhx_chronicliver':     ['K70', 'K71', 'K72', 'K73', 'K74', 'K75.4', 'K76.0', 'K76.6'],
    'pmhx_stroke':           ['I60', 'I61', 'I62', 'I63', 'I64', 'I65', 'I66', 'I67'],
    'pmhx_chf':              ['I50', 'I42', 'I43'],
    'pmhx_dementia':         ['F01', 'F02', 'F03', 'G30', 'G31.0', 'G31.83'],
    'pmhx_obesity':          ['E66', 'Z68'],
    'pmhx_hiv':              ['B20', 'Z21'],
    'pmhx_immunosuppressed': ['D80', 'D81', 'D82', 'D83', 'D84', 'Z79.6', 'Z79.62', 'Z79.63', 'Z94'],
    'pmhx_smoking':          ['F17', 'Z87.891'],
    'pmhx_mentalhealth':     ['F20', 'F25', 'F30', 'F31', 'F32', 'F33'],
    'pmhx_cystic_fibrosis':  ['E84'],
    'pmhx_tuberculosis':     ['A15', 'A16', 'A17', 'A18', 'A19'],
    'pmhx_sickle_cell':      ['D57'],
    'pmhx_substance_use':    ['F10', 'F11', 'F12', 'F13', 'F14', 'F15', 'F16', 'F18', 'F19'],
    'pmhx_atrialfib':        ['I48'],
}
PMHX_COLS = list(COMORBIDITY_DICT.keys())

def codes_to_like(codes):
    return ' OR '.join(f"icd_code LIKE '{c.replace(chr(46), '')}%'" for c in codes)

def build_pmhx_cases(d):
    return ',\n'.join(
        f"    MAX(CASE WHEN {codes_to_like(codes)} THEN 1 ELSE 0 END) AS {name}"
        for name, codes in d.items()
    )

def build_comorbidity_condition(d):
    seen, conds = set(), []
    for codes in d.values():
        for c in codes:
            cond = f"icd_code LIKE '{c.replace(chr(46), '')}%'"
            if cond not in seen:
                conds.append(cond)
                seen.add(cond)
    return '\n        OR '.join(conds)

# ── Extracción PMHX por ingreso ──────────────────────────────────────────────
pmhx_features = spark.sql(f"""
    SELECT
        hadm_id AS admission_id,
{build_pmhx_cases(COMORBIDITY_DICT)}
    FROM diagnoses_icd
    WHERE icd_version = 10
    GROUP BY hadm_id
""")
pmhx_features.createOrReplaceTempView('pmhx_features')
print(f'pmhx_features: {pmhx_features.count()} ingresos')

pmhx_features: 60973 ingresos


In [11]:
# ── Criterios de inclusión v6.1 ───────────────────────────────────────────────
# Se usa el criterio amplio de v6: U071 en cualquier posición seq_num.
# No hay restricción sobre el diagnóstico principal.

base_stays_v61 = spark.sql("""
    SELECT DISTINCT i.subject_id, i.admission_id, i.stay_id,
           i.length_of_stay, i.admittime
    FROM icu_stays_data i
    INNER JOIN diagnoses_icd d
        ON i.admission_id = d.hadm_id
    WHERE d.icd_code = 'U071'
""")
base_stays_v61.createOrReplaceTempView('base_stays_v61')
n_v61 = base_stays_v61.count()
print(f'Stays elegibles v6.1: {n_v61}  (equivalente a v6; v7 restringía a seq_num=1 o comorbilidad primaria)')


Stays elegibles v7: 983  (v6 tenía 3908 con cualquier posición de U071)


## 10. Construcción del dataset `all_stays` (Spark)

In [12]:
dataset_v61_all = (
    base_stays_v61
    .join(vital_features_v61, on=['subject_id', 'admission_id', 'stay_id'], how='left')
    .join(lab_features_v61,   on=['subject_id', 'admission_id', 'stay_id'], how='left')
    .join(procedure_flags,   on=['subject_id', 'admission_id', 'stay_id'], how='left')
    .join(pmhx_features,     on=['admission_id'],                           how='left')
    .join(
        patients_data.select('subject_id', 'admission_id', 'gender', 'anchor_age', 'dod_within_30_days'),
        on=['subject_id', 'admission_id'], how='left'
    )
    .join(
        hosp_adm_data.select('subject_id', 'admission_id', 'marital_status', 'race'),
        on=['subject_id', 'admission_id'], how='left'
    )
)

# Procedimientos y PMHX: NaN → 0 (no realizado / no presente)
for col_name in PROCEDURE_COLS + PMHX_COLS:
    dataset_v61_all = dataset_v61_all.fillna({col_name: 0})

dataset_v61_all.createOrReplaceTempView('dataset_v61_all')
print(f'dataset_v61_all: {dataset_v61_all.count()} filas | {len(dataset_v61_all.columns)} columnas')

dataset_v7_all: 983 filas | 163 columnas


In [13]:
# Checkpoint parquet
import os
os.makedirs('../data/processed', exist_ok=True)
dataset_v61_all.write.mode('overwrite').parquet('../data/processed/covid_icu_dataset_v61_all_stays.parquet')
print('Guardado: covid_icu_dataset_v61_all_stays.parquet')

Guardado: covid_icu_dataset_v7_all_stays.parquet


---
## 11. Post-procesamiento en Pandas

A partir de aquí se usa pandas.

In [14]:
import pandas as pd
import numpy as np

os.makedirs('../datasets/v6_1', exist_ok=True)

print('Cargando parquet all_stays...')
df = pd.read_parquet('../data/processed/covid_icu_dataset_v61_all_stays.parquet')
print(f'Shape: {df.shape}')

Cargando parquet all_stays...
Shape: (983, 163)


## 12. [NUEVO v7] Cálculo de deltas (last − first)

`val_delta > 0` → la variable subió en las 72h  
`val_delta < 0` → la variable bajó  
`val_delta = NaN` → no se midió (sin al menos dos lecturas)

In [15]:
print('Calculando deltas (last - first)...')

for _, name in LAB_ITEMS:
    df[f'lab_{name}_val_delta'] = df[f'lab_{name}_val_last'] - df[f'lab_{name}_val_first']

for _, name, *_ in CHART_ITEMS:
    df[f'vital_{name}_val_delta'] = df[f'vital_{name}_val_last'] - df[f'vital_{name}_val_first']

# Ejemplo de distribución de deltas clave
print('\nDistribución de deltas clave (no-nulos):')
for col in ['lab_creatinine_val_delta', 'lab_glucose_val_delta',
            'vital_spo2_val_delta', 'vital_hr_val_delta']:
    if col in df.columns:
        s = df[col].dropna()
        print(f'  {col:<30}: n={len(s):4d} | media={s.mean():+.2f} | p25={s.quantile(.25):+.2f} | p75={s.quantile(.75):+.2f}')

Calculando deltas (last - first)...

Distribución de deltas clave (no-nulos):
  lab_creatinine_val_delta      : n= 963 | media=-0.08 | p25=-0.14 | p75=+0.08
  lab_glucose_val_delta         : n= 970 | media=-5.50 | p25=-32.00 | p75=+30.00
  vital_spo2_val_delta          : n= 860 | media=+0.39 | p25=-2.00 | p75=+3.00
  vital_hr_val_delta            : n= 861 | media=-13.83 | p25=-24.00 | p75=+0.00


## 13. SOFA parcial (5/6 componentes)

Sin GCS. Para SOFA se usan los valores más graves del período:
- Respiratorio: `min` SpO₂ / PaO₂ (peor oxigenación)
- Coagulación: `min` plaquetas
- Hepático: `max` bilirrubina  
- Cardiovascular: PAM calculada desde `min` SBP/DBP
- Renal: `max` creatinina

In [16]:
def compute_map(row):
    """PAM = (PAS + 2·PAD) / 3. Prioriza no invasiva; fallback a línea arterial."""
    sbp = row.get('vital_sbp_val_min');   dbp = row.get('vital_dbp_val_min')
    if pd.notna(sbp) and pd.notna(dbp):
        return (sbp + 2 * dbp) / 3
    sbp_l = row.get('vital_sbp_line_val_min'); dbp_l = row.get('vital_dbp_line_val_min')
    if pd.notna(sbp_l) and pd.notna(dbp_l):
        return (sbp_l + 2 * dbp_l) / 3
    return np.nan


def compute_sofa_partial(row):
    score, n = 0, 0

    # 1. Respiratorio
    is_vent = row.get('is_mechanical_ventilation', 0) == 1
    fio2 = 0.50 if is_vent else 0.21
    pf = None
    if pd.notna(row.get('lab_pao2_val_min')) and row['lab_pao2_val_min'] > 0:
        pf = row['lab_pao2_val_min'] / fio2
    elif pd.notna(row.get('vital_spo2_val_min')):
        pf = row['vital_spo2_val_min'] / fio2
    if pf is not None:
        n += 1
        score += 0 if pf >= 400 else 1 if pf >= 300 else 2 if pf >= 200 else 3 if pf >= 100 else 4

    # 2. Coagulación
    plt = row.get('lab_platelets_val_min')
    if pd.notna(plt):
        n += 1
        score += 0 if plt >= 150 else 1 if plt >= 100 else 2 if plt >= 50 else 3 if plt >= 20 else 4

    # 3. Hepático
    bili = row.get('lab_bilirubin_val_max')
    if pd.notna(bili):
        n += 1
        score += 0 if bili < 1.2 else 1 if bili < 2.0 else 2 if bili < 6.0 else 3 if bili < 12.0 else 4

    # 4. Cardiovascular (PAM; sin vasopresores)
    map_val = compute_map(row)
    if pd.notna(map_val):
        n += 1
        score += 0 if map_val >= 70 else 1

    # 5. Neurológico: NO DISPONIBLE (sin GCS)

    # 6. Renal
    creat = row.get('lab_creatinine_val_max')
    if pd.notna(creat):
        n += 1
        score += 0 if creat < 1.2 else 1 if creat < 2.0 else 2 if creat < 3.5 else 3 if creat < 5.0 else 4

    return pd.Series({'sofa_partial': score if n > 0 else np.nan, 'sofa_n_components': n})


print('Calculando SOFA parcial...')
sofa_result = df.apply(compute_sofa_partial, axis=1)
df = pd.concat([df, sofa_result], axis=1)
print(f"Cobertura: {df['sofa_partial'].notna().mean():.1%}")
print(df['sofa_partial'].describe().round(2))

Calculando SOFA parcial...
Cobertura: 99.8%
count    981.00
mean       3.22
std        2.58
min        0.00
25%        1.00
50%        3.00
75%        5.00
max       14.00
Name: sofa_partial, dtype: float64


## 14. APACHE II parcial (11/12 componentes)

Sin GCS ni puntos por enfermedad crónica. Para APACHE II se usa el peor valor del período
(max para variables donde el exceso es peor; min para variables donde el déficit es peor).

In [17]:
def compute_apache2_partial(row):
    score, n = 0, 0

    # 1. Temperatura (°F → °C, peor = max)
    temp_f = row.get('vital_temp_val_max')
    if pd.notna(temp_f):
        t = (temp_f - 32) * 5 / 9; n += 1
        score += 4 if t >= 41 else 1 if t >= 39 else 0 if t >= 36 else 1 if t >= 34 else 2 if t >= 32 else 3 if t >= 30 else 4

    # 2. PAM (peor = mín)
    map_val = compute_map(row)
    if pd.notna(map_val):
        n += 1
        score += 4 if map_val >= 160 else 3 if map_val >= 130 else 2 if map_val >= 110 else 0 if map_val >= 70 else 2 if map_val >= 50 else 4

    # 3. FC (peor = max)
    hr = row.get('vital_hr_val_max')
    if pd.notna(hr):
        n += 1
        score += 4 if hr >= 180 else 3 if hr >= 140 else 2 if hr >= 110 else 0 if hr >= 70 else 2 if hr >= 55 else 3 if hr >= 40 else 4

    # 4. FR (peor = max)
    rr = row.get('vital_rr_val_max')
    if pd.notna(rr):
        n += 1
        score += 4 if rr >= 50 else 3 if rr >= 35 else 1 if rr >= 25 else 0 if rr >= 12 else 1 if rr >= 10 else 2 if rr >= 6 else 4

    # 5. Oxigenación
    pao2 = row.get('lab_pao2_val_min'); spo2 = row.get('vital_spo2_val_min')
    is_vent = row.get('is_mechanical_ventilation', 0) == 1
    if pd.notna(pao2) and not is_vent:
        n += 1
        score += 0 if pao2 >= 70 else 1 if pao2 >= 61 else 3 if pao2 >= 55 else 4
    elif pd.notna(spo2):
        n += 1
        score += 0 if spo2 >= 95 else 1 if spo2 >= 91 else 3 if spo2 >= 85 else 4

    # 6. pH (peor = mín)
    ph = row.get('lab_ph_val_min')
    if pd.notna(ph):
        n += 1
        score += 4 if ph >= 7.70 else 3 if ph >= 7.60 else 1 if ph >= 7.50 else 0 if ph >= 7.33 else 2 if ph >= 7.25 else 3 if ph >= 7.15 else 4

    # 7. Sodio (peor de mín o máx)
    na_min = row.get('lab_sodium_val_min'); na_max = row.get('lab_sodium_val_max')
    na = None
    if pd.notna(na_min) and pd.notna(na_max):
        na = na_min if na_min < 130 else na_max
    elif pd.notna(na_max): na = na_max
    elif pd.notna(na_min): na = na_min
    if na is not None:
        n += 1
        score += 4 if na >= 180 else 3 if na >= 160 else 2 if na >= 155 else 1 if na >= 150 else 0 if na >= 130 else 2 if na >= 120 else 3 if na >= 111 else 4

    # 8. Potasio (peor de mín o máx)
    k_min = row.get('lab_potassium_val_min'); k_max = row.get('lab_potassium_val_max')
    k = None
    if pd.notna(k_min) and pd.notna(k_max):
        k = k_max if k_max >= 5.5 else k_min
    elif pd.notna(k_max): k = k_max
    elif pd.notna(k_min): k = k_min
    if k is not None:
        n += 1
        score += 4 if k >= 7.0 else 3 if k >= 6.0 else 1 if k >= 5.5 else 0 if k >= 3.5 else 1 if k >= 3.0 else 2 if k >= 2.5 else 4

    # 9. Creatinina (peor = máx; ×2 si CRRT)
    creat = row.get('lab_creatinine_val_max')
    if pd.notna(creat):
        n += 1
        pts = 4 if creat >= 3.5 else 3 if creat >= 2.0 else 2 if creat >= 1.5 else 0 if creat >= 0.6 else 2
        if row.get('is_crrt', 0) == 1:
            pts = min(pts * 2, 4)
        score += pts

    # 10. Hematocrito (peor de mín o máx)
    hct = row.get('lab_hematocrit_val_max'); hct_min = row.get('lab_hematocrit_val_min')
    pts_hi, pts_lo = 0, 0
    if pd.notna(hct):
        pts_hi = 4 if hct >= 60 else 2 if hct >= 50 else 1 if hct >= 46 else 0
    if pd.notna(hct_min):
        pts_lo = 4 if hct_min < 20 else 2 if hct_min < 30 else 0
    if pd.notna(hct) or pd.notna(hct_min):
        n += 1; score += max(pts_hi, pts_lo)

    # 11. WBC (peor = máx)
    wbc = row.get('lab_wbc_val_max')
    if pd.notna(wbc):
        n += 1
        score += 4 if wbc >= 40 else 2 if wbc >= 20 else 1 if wbc >= 15 else 0 if wbc >= 3 else 2 if wbc >= 1 else 4

    # 12. GCS: NO DISPONIBLE

    # 13. Edad
    age = row.get('anchor_age')
    if pd.notna(age):
        n += 1
        score += 0 if age < 45 else 2 if age < 55 else 3 if age < 65 else 5 if age < 75 else 6

    return pd.Series({'apache2_partial': score if n >= 3 else np.nan, 'apache2_n_components': n})


print('Calculando APACHE II parcial...')
apache_result = df.apply(compute_apache2_partial, axis=1)
df = pd.concat([df, apache_result], axis=1)
print(f"Cobertura: {df['apache2_partial'].notna().mean():.1%}")
print(df['apache2_partial'].describe().round(2))

Calculando APACHE II parcial...
Cobertura: 99.8%
count    981.00
mean      12.62
std        5.20
min        0.00
25%        9.00
50%       12.00
75%       16.00
max       29.00
Name: apache2_partial, dtype: float64


In [18]:
# Validación clínica
df['_died'] = df['dod_within_30_days'].notna().astype(int)
for score_col in ['sofa_partial', 'apache2_partial']:
    alive = df.loc[df['_died'] == 0, score_col].dropna()
    dead  = df.loc[df['_died'] == 1, score_col].dropna()
    ok = '✅' if dead.median() > alive.median() else '⚠️'
    print(f'{ok} {score_col}: vivos={alive.median():.1f} vs muertos={dead.median():.1f}')
df.drop(columns=['_died'], inplace=True)

✅ sofa_partial: vivos=2.0 vs muertos=4.0
✅ apache2_partial: vivos=11.0 vs muertos=15.5


## 15. [NUEVO v7] Variables binarias clínicas

Para maximizar el número de features binarias útiles para XGBoost.

**Convención NaN:**
- Variables de laboratorio/vitales: `fillna(valor_normal)` → 0 cuando no medida. El indicador `_measured` da contexto.
- Scores (SOFA, APACHE II): `NaN` cuando no calculable — XGBoost lo gestiona nativamente.

| Variable | Umbral | Referencia |
|---|---|---|
| `bin_sofa_ge2` | SOFA ≥ 2 | Disfunción orgánica (Sepsis-3) |
| `bin_apache2_ge10` | APACHE II ≥ 10 | Mortalidad hospitalaria ~15-25% |
| `bin_spo2_lt92` | SpO₂ mín < 92% | Hipoxemia moderada |
| `bin_fever` | Temp máx > 100.4°F | Fiebre (38°C) |
| `bin_tachycardia` | FC máx > 100 lpm | Taquicardia |
| `bin_tachypnea` | FR máx > 20 rpm | Taquipnea |
| `bin_map_lt65` | PAM mín < 65 mmHg | Shock |
| `bin_obese` | BMI ≥ 30 | Obesidad |
| `bin_creatinine_hi` | Creatinina máx > 1.2 | Disfunción renal |
| `bin_bilirubin_hi` | Bilirrubina máx > 1.2 | Disfunción hepática |
| `bin_thrombocytopenia` | Plaquetas mín < 150 | Trombocitopenia |
| `bin_hyperglycemia` | Glucosa máx > 180 | Hiperglucemia |
| `bin_bun_hi` | BUN máx > 25 | Azoemia |
| `bin_lactate_hi` | Lactato máx > 2.0 | Hipoperfusión |
| `bin_hyponatremia` | Na mín < 135 | Hiponatremia |
| `bin_hypernatremia` | Na máx > 145 | Hipernatremia |
| `bin_hypokalemia` | K mín < 3.5 | Hipopotasemia |
| `bin_hyperkalemia` | K máx > 5.0 | Hiperpotasemia |
| `bin_anemia` | Hct mín < 30% | Anemia |
| `bin_leukocytosis` | WBC máx > 12 | Leucocitosis |
| `bin_leukopenia` | WBC mín < 4 | Leucopenia |
| `bin_pao2_lt200` | PaO₂ mín < 200 mmHg | SDRA moderado-severo |
| `bin_age_ge65` | Edad ≥ 65 | Anciano |
| `bin_age_ge75` | Edad ≥ 75 | Anciano avanzado |

In [19]:
print('Creando variables binarias clínicas...')

# ── Scores (NaN cuando no calculable) ───────────────────────────────────────
df['bin_sofa_ge2']     = (df['sofa_partial']     >= 2).where(df['sofa_partial'].notna()).astype(float)
df['bin_apache2_ge10'] = (df['apache2_partial'] >= 10).where(df['apache2_partial'].notna()).astype(float)

# ── Signos vitales (fillna → no anormal cuando no medido) ───────────────────
df['bin_spo2_lt92']   = (df['vital_spo2_val_min'].fillna(100) < 92).astype(int)
df['bin_fever']       = (df['vital_temp_val_max'].fillna(98.6) > 100.4).astype(int)
df['bin_tachycardia'] = (df['vital_hr_val_max'].fillna(70) > 100).astype(int)
df['bin_tachypnea']   = (df['vital_rr_val_max'].fillna(16) > 20).astype(int)
df['bin_obese']       = (df['vital_bmi_val_first'].fillna(0) >= 30).astype(int)

map_series = df.apply(compute_map, axis=1)
df['bin_map_lt65'] = (map_series.fillna(999) < 65).astype(int)

# ── Laboratorio ──────────────────────────────────────────────────────────────
df['bin_creatinine_hi']    = (df['lab_creatinine_val_max'].fillna(0)   > 1.2).astype(int)
df['bin_bilirubin_hi']     = (df['lab_bilirubin_val_max'].fillna(0)    > 1.2).astype(int)
df['bin_thrombocytopenia'] = (df['lab_platelets_val_min'].fillna(999)  < 150).astype(int)
df['bin_hyperglycemia']    = (df['lab_glucose_val_max'].fillna(0)      > 180).astype(int)
df['bin_bun_hi']           = (df['lab_bun_val_max'].fillna(0)          > 25).astype(int)
df['bin_lactate_hi']       = (df['lab_lactate_val_max'].fillna(0)      > 2.0).astype(int)
df['bin_hyponatremia']     = (df['lab_sodium_val_min'].fillna(140)     < 135).astype(int)
df['bin_hypernatremia']    = (df['lab_sodium_val_max'].fillna(140)     > 145).astype(int)
df['bin_hypokalemia']      = (df['lab_potassium_val_min'].fillna(4.0)  < 3.5).astype(int)
df['bin_hyperkalemia']     = (df['lab_potassium_val_max'].fillna(4.0)  > 5.0).astype(int)
df['bin_anemia']           = (df['lab_hematocrit_val_min'].fillna(999) < 30).astype(int)
df['bin_leukocytosis']     = (df['lab_wbc_val_max'].fillna(0)          > 12).astype(int)
df['bin_leukopenia']       = (df['lab_wbc_val_min'].fillna(999)        < 4).astype(int)
df['bin_pao2_lt200']       = (df['lab_pao2_val_min'].fillna(999)       < 200).astype(int)

# ── Demografía ───────────────────────────────────────────────────────────────
df['bin_age_ge65'] = (df['anchor_age'] >= 65).astype(int)
df['bin_age_ge75'] = (df['anchor_age'] >= 75).astype(int)

BIN_COLS = [
    'bin_sofa_ge2', 'bin_apache2_ge10',
    'bin_spo2_lt92', 'bin_fever', 'bin_tachycardia', 'bin_tachypnea',
    'bin_map_lt65', 'bin_obese',
    'bin_creatinine_hi', 'bin_bilirubin_hi', 'bin_thrombocytopenia',
    'bin_hyperglycemia', 'bin_bun_hi', 'bin_lactate_hi',
    'bin_hyponatremia', 'bin_hypernatremia',
    'bin_hypokalemia', 'bin_hyperkalemia',
    'bin_anemia', 'bin_leukocytosis', 'bin_leukopenia',
    'bin_pao2_lt200',
    'bin_age_ge65', 'bin_age_ge75',
]
print(f'{len(BIN_COLS)} variables binarias creadas.')

# Distribución por mortalidad
died = df['dod_within_30_days'].notna()
print('\nPrevalencia (total | vivos | muertos):')
for col in BIN_COLS:
    t = df[col].mean(); a = df.loc[~died, col].mean(); d = df.loc[died, col].mean()
    print(f'  {col:<26}: {t:.1%} | {a:.1%} | {d:.1%}')

Creando variables binarias clínicas...
24 variables binarias creadas.

Prevalencia (total | vivos | muertos):
  bin_sofa_ge2              : 68.1% | 62.0% | 85.0%
  bin_apache2_ge10          : 70.6% | 64.2% | 88.5%
  bin_spo2_lt92             : 75.3% | 72.5% | 83.1%
  bin_fever                 : 25.2% | 24.6% | 26.9%
  bin_tachycardia           : 51.1% | 48.1% | 59.2%
  bin_tachypnea             : 85.4% | 84.6% | 87.3%
  bin_map_lt65              : 39.7% | 34.3% | 54.6%
  bin_obese                 : 31.5% | 33.1% | 27.3%
  bin_creatinine_hi         : 38.4% | 32.1% | 55.8%
  bin_bilirubin_hi          : 9.3% | 7.5% | 14.2%
  bin_thrombocytopenia      : 22.7% | 18.9% | 33.1%
  bin_hyperglycemia         : 54.6% | 52.1% | 61.5%
  bin_bun_hi                : 57.7% | 50.5% | 77.7%
  bin_lactate_hi            : 7.2% | 6.2% | 10.0%
  bin_hyponatremia          : 26.9% | 28.1% | 23.5%
  bin_hypernatremia         : 11.0% | 8.4% | 18.1%
  bin_hypokalemia           : 18.2% | 18.4% | 17.7%
  bin_hyper

## 16. Indicadoras de missingness y severity_score

In [20]:
# Indicadora de missingness: 1 si se midió algún valor en la ventana 72h
# Se usa _val_max como referencia (NaN ↔ no medido)
print('Creando indicadoras de missingness...')

for _, name in LAB_ITEMS:
    df[f'lab_{name}_measured'] = df[f'lab_{name}_val_max'].notna().astype(int)

# Para vitales: excluir line BP (se elimina al final)
for _, name, *_ in CHART_ITEMS:
    if 'line' not in name:
        df[f'vital_{name}_measured'] = df[f'vital_{name}_val_max'].notna().astype(int)

print('\nCobertura de labs (% medidos):')
lab_cov = {name: df[f'lab_{name}_measured'].mean() for _, name in LAB_ITEMS}
for name, v in sorted(lab_cov.items(), key=lambda x: -x[1]):
    bar = '█' * int(v * 20) + '░' * (20 - int(v * 20))
    print(f'  lab_{name:<20} {bar} {v:.1%}')

Creando indicadoras de missingness...

Cobertura de labs (% medidos):
  lab_glucose              ███████████████████░ 98.7%
  lab_potassium            ███████████████████░ 98.1%
  lab_creatinine           ███████████████████░ 98.0%
  lab_bun                  ███████████████████░ 98.0%
  lab_sodium               ███████████████████░ 98.0%
  lab_hematocrit           ███████████████████░ 97.5%
  lab_platelets            ███████████████████░ 97.5%
  lab_wbc                  ███████████████████░ 97.5%
  lab_bilirubin            ████████████████░░░░ 83.1%
  lab_crp                  ████████████████░░░░ 82.8%
  lab_neutrophils          ██████████████░░░░░░ 72.6%
  lab_lymphocytes          █████████████░░░░░░░ 66.2%
  lab_ferritin             ████████████░░░░░░░░ 62.7%
  lab_pao2                 █████████░░░░░░░░░░░ 46.6%
  lab_pt                   ███████░░░░░░░░░░░░░ 37.2%
  lab_dimer                ██████░░░░░░░░░░░░░░ 33.4%
  lab_lactate              ████░░░░░░░░░░░░░░░░ 22.4%
  lab_peep  

In [21]:
def compute_severity_score_v7(row, low_coverage_cols):
    """Severity score compuesto v7 — usa nuevos nombres de columnas."""
    components, weights = [], []

    def add(val, w, norm_fn):
        if pd.notna(val):
            components.append(np.clip(norm_fn(val), 0, 1))
            weights.append(w)

    # Respiratorio
    add(row.get('vital_spo2_val_min'), 3, lambda x: (100 - x) / 100)
    if 'lab_pao2' not in low_coverage_cols:
        add(row.get('lab_pao2_val_min'), 3, lambda x: 1 - x / 500)
    add(row.get('vital_rr_val_max'), 2, lambda x: (x - 12) / 28)

    # Hemodinámico
    add(row.get('vital_sbp_val_min'), 2, lambda x: 1 - x / 120)

    # Metabólico
    if 'lab_lactate' not in low_coverage_cols:
        add(row.get('lab_lactate_val_max'), 3, lambda x: x / 10)

    # Acidosis
    if 'lab_ph' not in low_coverage_cols:
        add(row.get('lab_ph_val_min'), 3, lambda x: (7.45 - x) / 0.30)

    # Renal
    if 'lab_creatinine' not in low_coverage_cols:
        add(row.get('lab_creatinine_val_max'), 2, lambda x: x / 10)

    # Electrolitos
    if 'lab_sodium' not in low_coverage_cols:
        na_min = row.get('lab_sodium_val_min'); na_max = row.get('lab_sodium_val_max')
        na_worst = None
        if pd.notna(na_min) and pd.notna(na_max):
            na_worst = na_min if abs(na_min - 140) > abs(na_max - 140) else na_max
        elif pd.notna(na_min): na_worst = na_min
        elif pd.notna(na_max): na_worst = na_max
        if na_worst is not None:
            add(abs(na_worst - 140), 1, lambda x: x / 30)

    # Hepático
    if 'lab_bilirubin' not in low_coverage_cols:
        add(row.get('lab_bilirubin_val_max'), 1, lambda x: x / 20)

    # Coagulación
    if 'lab_platelets' not in low_coverage_cols:
        add(row.get('lab_platelets_val_min'), 2, lambda x: 1 - x / 400)
    if 'lab_dimer' not in low_coverage_cols:
        add(row.get('lab_dimer_val_max'), 2, lambda x: x / 10)

    # Inflamación
    if 'lab_crp' not in low_coverage_cols:
        add(row.get('lab_crp_val_max'), 2, lambda x: x / 300)
    if 'lab_lymphocytes' not in low_coverage_cols:
        add(row.get('lab_lymphocytes_val_min'), 2, lambda x: 1 - x / 2000)
    if 'lab_ferritin' not in low_coverage_cols:
        add(row.get('lab_ferritin_val_max'), 2, lambda x: x / 5000)
    if 'lab_troponin' not in low_coverage_cols:
        add(row.get('lab_troponin_val_max'), 2, lambda x: x / 10)

    # Procedimientos invasivos
    for proc, w in [('is_ecmo', 5), ('is_mechanical_ventilation', 4),
                    ('is_prone_position', 3), ('is_crrt', 3),
                    ('has_hemodialysis', 3), ('is_intubated', 2)]:
        val = row.get(proc)
        if pd.notna(val):
            components.append(float(val)); weights.append(w)

    return np.average(components, weights=weights) if components else np.nan


# Umbral de baja cobertura (>70% missings)
lab_coverage = {name: df[f'lab_{name}_measured'].mean() for _, name in LAB_ITEMS}
low_coverage_cols = {name for name, cov in lab_coverage.items() if cov < 0.30}
print(f'Labs con baja cobertura (<30%, excluidos del severity_score): {low_coverage_cols}')

print('\nCalculando severity_score v7...')
df['severity_score'] = df.apply(
    lambda r: compute_severity_score_v7(r, low_coverage_cols), axis=1
)
print(f"severity_score cobertura: {df['severity_score'].notna().mean():.1%}")

Labs con baja cobertura (<30%, excluidos del severity_score): {'troponin', 'ph', 'lactate', 'tidal_volume', 'peep'}

Calculando severity_score v7...
severity_score cobertura: 100.0%


## 17. Limpieza final de columnas

In [22]:
# Columnas line BP: solo se usaron para MAP, se eliminan del dataset final
LINE_BP_COLS = [
    'vital_sbp_line_val_first', 'vital_sbp_line_val_last',
    'vital_sbp_line_val_min',   'vital_sbp_line_val_max',  'vital_sbp_line_val_delta',
    'vital_dbp_line_val_first', 'vital_dbp_line_val_last',
    'vital_dbp_line_val_min',   'vital_dbp_line_val_max',  'vital_dbp_line_val_delta',
]

DROP_ALWAYS = LINE_BP_COLS + [
    'length_of_stay',            # leakage temporal
    'is_dnr',                    # variable dependiente del outcome
    'sofa_n_components',         # auxiliar de cálculo
    'apache2_n_components',      # auxiliar de cálculo
]

df.drop(columns=[c for c in DROP_ALWAYS if c in df.columns], inplace=True)

print(f'Shape final all_stays: {df.shape}')
print(f'\nColumnas ({len(df.columns)}):')
for i, c in enumerate(df.columns, 1):
    print(f'  {i:>3}. {c}')

Shape final all_stays: (983, 236)

Columnas (236):
    1. subject_id
    2. admission_id
    3. stay_id
    4. admittime
    5. vital_temp_val_first
    6. vital_temp_val_last
    7. vital_temp_val_min
    8. vital_temp_val_max
    9. vital_sbp_val_first
   10. vital_sbp_val_last
   11. vital_sbp_val_min
   12. vital_sbp_val_max
   13. vital_dbp_val_first
   14. vital_dbp_val_last
   15. vital_dbp_val_min
   16. vital_dbp_val_max
   17. vital_spo2_val_first
   18. vital_spo2_val_last
   19. vital_spo2_val_min
   20. vital_spo2_val_max
   21. vital_hr_val_first
   22. vital_hr_val_last
   23. vital_hr_val_min
   24. vital_hr_val_max
   25. vital_rr_val_first
   26. vital_rr_val_last
   27. vital_rr_val_min
   28. vital_rr_val_max
   29. vital_bmi_val_first
   30. vital_bmi_val_last
   31. vital_bmi_val_min
   32. vital_bmi_val_max
   33. lab_glucose_val_first
   34. lab_glucose_val_last
   35. lab_glucose_val_min
   36. lab_glucose_val_max
   37. lab_creatinine_val_first
   38. lab_crea

## 18. Compactación: una estancia por paciente

**Opción B:** mayor `severity_score` (estancia más grave)  
**Opción C:** última estancia (`admittime` más reciente)

In [23]:
print(f'Pacientes únicos:        {df["subject_id"].nunique()}')
print(f'Estancias totales:       {len(df)}')

# ── Opción B: estancia más grave según SOFA + APACHE II (normalizados) ───────
# severity_score ELIMINADO de Opción B: ni criterio de selección ni feature.
#
# Normalización:
#   SOFA máx teórico    = 20  (5 componentes × 4 pts, sin GCS)
#   APACHE II máx parcial = 51  (sin GCS ni enfermedad crónica)
#
# Combined score = SOFA/20 + APACHE·II/51  →  rango [0, 2.0]
# NaN → 0 para la selección (estancias sin score quedan al fondo del ranking)
df['_sofa_norm']   = df['sofa_partial'].fillna(0) / 20
df['_apache_norm'] = df['apache2_partial'].fillna(0) / 51
df['_optB_score']  = df['_sofa_norm'] + df['_apache_norm']

df_optB = (
    df.sort_values(by=['subject_id', '_optB_score'], ascending=[True, False])
      .drop_duplicates(subset=['subject_id'], keep='first')
      .drop(columns=['_sofa_norm', '_apache_norm', '_optB_score', 'severity_score'],
            errors='ignore')
)
print(f'OptionB: {len(df_optB)} pacientes  '
      f'(criterio: SOFA/20 + APACHE·II/51 | severity_score eliminado)')

# ── Opción C: última estancia por admittime ───────────────────────────────────
# severity_score se conserva como feature adicional en Opción C.
df_optC = (
    df.sort_values(by=['subject_id', 'admittime'], ascending=[True, False])
      .drop_duplicates(subset=['subject_id'], keep='first')
)
print(f'OptionC: {len(df_optC)} pacientes  (con severity_score)')

Pacientes únicos:        862
Estancias totales:       983
OptionB: 862 pacientes  (criterio: SOFA/20 + APACHE·II/51 | severity_score eliminado)
OptionC: 862 pacientes  (con severity_score)


## 19. Validación del dataset v7

In [24]:
print('=' * 60)
print('VALIDACIÓN DATASET V6.1')
print('=' * 60)

for name, d in [('OptionB', df_optB), ('OptionC', df_optC)]:
    print(f'\n── {name} ───────────────────────────────────────────────')
    print(f'  Shape:          {d.shape}')
    print(f'  Mortalidad 30d: {d["dod_within_30_days"].notna().mean():.2%}')

    # severity_score solo en OptionC
    score_cols = ['sofa_partial', 'apache2_partial']
    if 'severity_score' in d.columns:
        score_cols.append('severity_score')

    for col in score_cols:
        cov = d[col].notna().mean()
        med = d[col].median()
        print(f'  {col:<25}: {cov:.1%} cobertura | mediana = {med:.2f}')

    for score in ['sofa_partial', 'apache2_partial']:
        alive = d.loc[d['dod_within_30_days'].isna(), score].dropna()
        dead  = d.loc[d['dod_within_30_days'].notna(), score].dropna()
        ok = '✅' if dead.median() > alive.median() else '⚠️'
        print(f'  {ok} {score}: vivos={alive.median():.1f} vs muertos={dead.median():.1f}')

    # Verificar severity_score
    if 'severity_score' in d.columns:
        print(f'  ✅ severity_score presente (OptionC)')
    else:
        print(f'  ✅ severity_score eliminado (OptionB — compactación por SOFA+APACHE·II)')

    # Verificar columnas clave
    missing_cols = [c for c in
        [f'lab_{nm}_val_delta' for _, nm in LAB_ITEMS[:3]] +
        [f'vital_{nm}_val_delta' for _, nm, *_ in CHART_ITEMS[:3]] +
        PMHX_COLS[:3] + BIN_COLS[:3]
        if c not in d.columns]
    if missing_cols:
        print(f'  ⚠️  Columnas faltantes: {missing_cols}')
    else:
        print(f'  ✅ Deltas, PMHX y binary features presentes')

    # Consistencia bin_sofa
    sub = d.loc[d['sofa_partial'].notna()]
    expected = (sub['sofa_partial'] >= 2).astype(float)
    assert (sub['bin_sofa_ge2'] == expected).all(), 'ERROR: bin_sofa_ge2 inconsistente'
    print(f'  ✅ bin_sofa_ge2 consistente con sofa_partial')

print('\n✅ Validación completada.')

VALIDACIÓN DATASET V7

── OptionB ───────────────────────────────────────────────
  Shape:          (862, 235)
  Mortalidad 30d: 25.87%
  sofa_partial             : 99.9% cobertura | mediana = 3.00
  apache2_partial          : 99.9% cobertura | mediana = 13.00
  ✅ sofa_partial: vivos=2.0 vs muertos=5.0
  ✅ apache2_partial: vivos=12.0 vs muertos=16.0
  ✅ severity_score eliminado (OptionB — compactación por SOFA+APACHE·II)
  ✅ Deltas, PMHX y binary features presentes
  ✅ bin_sofa_ge2 consistente con sofa_partial

── OptionC ───────────────────────────────────────────────
  Shape:          (862, 239)
  Mortalidad 30d: 26.10%
  sofa_partial             : 99.9% cobertura | mediana = 3.00
  apache2_partial          : 99.9% cobertura | mediana = 12.00
  severity_score           : 100.0% cobertura | mediana = 0.22
  ✅ sofa_partial: vivos=2.0 vs muertos=5.0
  ✅ apache2_partial: vivos=11.0 vs muertos=16.0
  ✅ severity_score presente (OptionC)
  ✅ Deltas, PMHX y binary features presentes
  ✅ bin_

## 20. Guardado de datasets v7

In [25]:
df_optB.to_parquet('../datasets/v6_1/dataset_v61_optionB.parquet', index=False, compression='snappy')
df_optB.to_csv('../datasets/v6_1/dataset_v61_optionB.csv', index=False)
print(f'✅ v6.1 OptionB guardado: {df_optB.shape}  (sin severity_score)')

df_optC.to_parquet('../datasets/v6_1/dataset_v61_optionC.parquet', index=False, compression='snappy')
df_optC.to_csv('../datasets/v6_1/dataset_v61_optionC.csv', index=False)
print(f'✅ v6.1 OptionC guardado: {df_optC.shape}  (con severity_score)')

n_lab  = len(LAB_ITEMS) * 5
n_vit  = len([x for x in CHART_ITEMS if 'line' not in x[1]]) * 5
n_proc = len(PROCEDURE_COLS) - 1   # -1 por is_dnr eliminado

print(f'\nResumen de features por opción:')
print(f'  {"Feature group":<35} {"OptionB":>8} {"OptionC":>8}')
print(f'  {"-"*51}')
print(f'  {"Labs (×5 agg: first/last/min/max/delta)":<35} {n_lab:>8} {n_lab:>8}')
print(f'  {"Vitales (×5 agg)":<35} {n_vit:>8} {n_vit:>8}')
print(f'  {"Indicadoras missingness":<35} {len(LAB_ITEMS)+len([x for x in CHART_ITEMS if "line" not in x[1]]):>8} {len(LAB_ITEMS)+len([x for x in CHART_ITEMS if "line" not in x[1]]):>8}')
print(f'  {"Procedimientos":<35} {n_proc:>8} {n_proc:>8}')
print(f'  {"PMHX comorbilidades":<35} {len(PMHX_COLS):>8} {len(PMHX_COLS):>8}')
print(f'  {"Variables binarias clínicas":<35} {len(BIN_COLS):>8} {len(BIN_COLS):>8}')
print(f'  {"SOFA + APACHE II (continuos)":<35} {"2":>8} {"2":>8}')
print(f'  {"Severity score":<35} {"—":>8} {"1":>8}')
print(f'  {"Demográficas":<35} {"4":>8} {"4":>8}')
print(f'\n  → datasets/v7/dataset_v61_optionB.parquet')
print(f'  → datasets/v7/dataset_v61_optionC.parquet')

✅ OptionB guardado: (862, 235)  (sin severity_score)
✅ OptionC guardado: (862, 239)  (con severity_score)

Resumen de features por opción:
  Feature group                        OptionB  OptionC
  ---------------------------------------------------
  Labs (×5 agg: first/last/min/max/delta)      105      105
  Vitales (×5 agg)                          35       35
  Indicadoras missingness                   28       28
  Procedimientos                            10       10
  PMHX comorbilidades                       22       22
  Variables binarias clínicas               24       24
  SOFA + APACHE II (continuos)               2        2
  Severity score                             —        1
  Demográficas                               4        4

  → datasets/v7/dataset_v7_optionB.parquet
  → datasets/v7/dataset_v7_optionC.parquet
